# Drivix MLDLC — Stage 3: Data Preprocessing & Feature Engineering

**Project:** Drivix Smart Parking Dynamic Pricing Engine  
**Objective:** Engineer cyclical temporal representations, non-linear transforms, and prevent leakage with Scikit-Learn pipelines.  
**Subsystem:** `ml_service`

---

## 1. Objectives & Preprocessing Plan
1. **Handling Missing Values:**
   - `f_occ_same_hour_last_week` has missing values in Week 1. We impute these using the median historical occupancy computed **strictly on the training split**.
2. **Cyclical Temporal Encoding:**
   - Hour of the day ($0–23$) and Day of the week ($0–6$) are cyclical. Encoding them linearly implies that hour 23 and hour 0 are maximally distant. We apply trigonometric sine and cosine transformations:
     $$\sin(\text{hour}) = \sin\left(\frac{2\pi \times \text{hour}}{24}\right), \quad \cos(\text{hour}) = \cos\left(\frac{2\pi \times \text{hour}}{24}\right)$$
     $$\sin(\text{dow}) = \sin\left(\frac{2\pi \times \text{dow}}{7}\right), \quad \cos(\text{dow}) = \cos\left(\frac{2\pi \times \text{dow}}{7}\right)$$
3. **Log Transformation:**
   - Right-skewed lead time: $\log(1 + \text{lead\_time\_min})$.
4. **Interaction Variables:**
   - $\text{est\_busy\_slots} = f\_busy\_frac\_window\_at\_quote \times f\_capacity$ (converts percentage to physical slot counts).
5. **Categorical Encoding:**
   - One-hot encoding for `f_location` (`drop='first'`).
6. **Strict Pipeline Isolation:**
   - All transformations are fitted strictly on the Training set (Weeks 1–12) to avoid validation/test set data leakage.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.append(os.path.join("..", "src"))
from feature_engineering import build_preprocessor_pipeline, FEATURE_COLUMNS_RAW, get_engineered_feature_names

DATA_DIR = os.path.join("..", "data", "processed")
train_df = pd.read_csv(os.path.join(DATA_DIR, "train_dataset.csv"))
valid_df = pd.read_csv(os.path.join(DATA_DIR, "valid_dataset.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test_dataset.csv"))

print(f"Loaded datasets from {DATA_DIR}:")
print(f" - Train: {train_df.shape}")
print(f" - Valid: {valid_df.shape}")
print(f" - Test:  {test_df.shape}")

## 2. Inspecting Missing Values & Imputation Strategy

In [ ]:
print("Missing values in Train set:")
print(train_df[FEATURE_COLUMNS_RAW].isnull().sum()[train_df[FEATURE_COLUMNS_RAW].isnull().sum() > 0])

print("\nMissing values in Valid set:")
print(valid_df[FEATURE_COLUMNS_RAW].isnull().sum()[valid_df[FEATURE_COLUMNS_RAW].isnull().sum() > 0])

print("\nMissing values in Test set:")
print(test_df[FEATURE_COLUMNS_RAW].isnull().sum()[test_df[FEATURE_COLUMNS_RAW].isnull().sum() > 0])

Notice that `valid_df` and `test_df` have **zero** missing values because they occur after Week 1! Only `train_df` contains initial cold-start NaNs. Median imputation fitted on training data safely handles this without leakage.

## 3. Fitting the Feature Engineering Pipeline
We fit the `ColumnTransformer` on `X_train` only.

In [ ]:
X_train = train_df[FEATURE_COLUMNS_RAW]
y_train = train_df['y_reserved_occupancy_rate']

X_valid = valid_df[FEATURE_COLUMNS_RAW]
y_valid = valid_df['y_reserved_occupancy_rate']

X_test = test_df[FEATURE_COLUMNS_RAW]
y_test = test_df['y_reserved_occupancy_rate']

# Build and fit pipeline (with scaling for linear models)
pipeline = build_preprocessor_pipeline(scale_numeric=True)
pipeline.fit(X_train)

# Transform splits
X_train_trans = pipeline.transform(X_train)
X_valid_trans = pipeline.transform(X_valid)
X_test_trans = pipeline.transform(X_test)

feature_names = get_engineered_feature_names(pipeline)
print(f"Engineered feature count: {len(feature_names)}")
print("Engineered feature names:")
for i, name in enumerate(feature_names, 1):
    print(f" {i:2d}. {name}")

## 4. Visualizing Cyclical Feature Representations
Let us verify the sine-cosine encoding of the 24-hour cycle.

In [ ]:
import matplotlib.pyplot as plt

hours = np.arange(24)
sin_h = np.sin(2 * np.pi * hours / 24.0)
cos_h = np.cos(2 * np.pi * hours / 24.0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(hours, sin_h, label='sin_hour', marker='o', color='#4f46e5')
ax1.plot(hours, cos_h, label='cos_hour', marker='s', color='#d97706')
ax1.set_title("Trigonometric Hour Cycle Encoding", fontsize=11, fontweight='bold')
ax1.set_xlabel("Clock Hour")
ax1.set_ylabel("Transformed Coordinate")
ax1.set_xticks(range(0, 24, 2))
ax1.legend()

ax2.scatter(sin_h, cos_h, c=hours, cmap='twilight', s=70)
for h in [0, 6, 12, 18]:
    ax2.annotate(f"{h}:00", (sin_h[h], cos_h[h]), textcoords="offset points", xytext=(8, 4), fontweight='bold')
ax2.set_title("Circular Representation (Hour 23 is Adjacent to Hour 0)", fontsize=11, fontweight='bold')
ax2.set_xlabel("sin_hour")
ax2.set_ylabel("cos_hour")
ax2.set_aspect('equal')

plt.tight_layout()
plt.show()

## 5. Serializing Processed Feature Matrices & Preprocessor
We save the fitted preprocessor artifact to `models/preprocessor.joblib` for reproducibility and inference.

In [ ]:
import joblib

MODEL_DIR = os.path.join("..", "models")
os.makedirs(MODEL_DIR, exist_ok=True)

preprocessor_path = os.path.join(MODEL_DIR, "preprocessor.joblib")
joblib.dump(pipeline, preprocessor_path)
print(f"Fitted preprocessor successfully saved to: {preprocessor_path}")